# CONFIG

In [1]:
import argparse
import os
import pandas as pd
import numpy as np

from dataclasses import dataclass
from maikol_utils.print_utils import print_separator


pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
@dataclass
class Configuration:
    """Configuration dataclass to hold application settings."""
    DATA_PATH: str = "data/scores"
    
    data_name: str = "A"
    clients_path: str = ""
    impostors_path: str = ""

    # ========================================
    threshold: float = 0.1
    

    def __post_init__(self):
        self.clients_path = os.path.join(self.DATA_PATH, f"scores{self.data_name}_clientes")
        self.impostors_path = os.path.join(self.DATA_PATH, f"scores{self.data_name}_impostores")


# CODE

## Load data

In [3]:
CONFIG_A = Configuration(data_name="A")
CONFIG_B = Configuration(data_name="B")

print(f"Data A at {CONFIG_A.clients_path} & {CONFIG_A.impostors_path}")
print(f"Data B at {CONFIG_B.clients_path} & {CONFIG_B.impostors_path}")

# Load A
df_clients_A = pd.read_csv(CONFIG_A.clients_path, sep=" ", header=None, names=["id", "score"])
df_impostors_A = pd.read_csv(CONFIG_A.impostors_path, sep=" ", header=None, names=["id", "score"])
df_clients_A["client"] = True
df_impostors_A["client"] = False
df_all_A = pd.concat([df_clients_A, df_impostors_A])

# Load B
df_clients_B = pd.read_csv(CONFIG_B.clients_path, sep=" ", header=None, names=["id", "score"])
df_impostors_B = pd.read_csv(CONFIG_B.impostors_path, sep=" ", header=None, names=["id", "score"])
df_clients_B["client"] = True
df_impostors_B["client"] = False
df_all_B = pd.concat([df_clients_B, df_impostors_B])

print("\nDataset A sample:")
print(df_all_A.sample(10))
print("\nDataset B sample:")
print(df_all_B.sample(10))


Data A at data/scores/scoresA_clientes & data/scores/scoresA_impostores
Data B at data/scores/scoresB_clientes & data/scores/scoresB_impostores

Dataset A sample:
        id     score  client
762   1004  0.002928   False
847   1005  0.063418    True
361   1030  0.015186   False
232   1027  0.253243    True
423   1013  0.004687   False
261   1002  0.141751    True
1069  1013  0.018161   False
1535  1013  0.008601   False
1066  1013  0.017079   False
114   1012  0.000000   False

Dataset B sample:
        id     score  client
844   1013  0.165389    True
91    1009  0.000000   False
538   1030  0.009238   False
67    1005  0.002833    True
777   1033  0.080717    True
186   1031  0.894850    True
1368  1005  0.117860    True
350   1011  0.014113   False
809   1011  0.071542   False
748   1030  0.013498   False


In [4]:
print("Dataset A:")
print(df_all_A.describe())
print("\nDataset B:")
print(df_all_B.describe())


Dataset A:
                id        score
count  2990.000000  2990.000000
mean   1020.000000     0.123788
std      13.530012     0.199723
min    1001.000000     0.000000
25%    1007.000000     0.020749
50%    1020.000000     0.048664
75%    1033.000000     0.106090
max    1039.000000     1.000000

Dataset B:
                id        score
count  2990.000000  2990.000000
mean   1020.000000     0.165057
std      13.530012     0.264623
min    1001.000000     0.000000
25%    1007.000000     0.003952
50%    1020.000000     0.040472
75%    1033.000000     0.175034
max    1039.000000     1.000000


## Compute

In [5]:
def compute_FP(df, threshold):
    impostors = df[~df["client"]]
    return impostors[impostors["score"] > threshold].shape[0] / impostors.shape[0]

def compute_FN(df, threshold):
    clients = df[df["client"]]
    return 1 - clients[clients["score"] <= threshold].shape[0] / clients.shape[0]


threshold = 0.1
print(f"Dataset A - FP: {compute_FP(df_all_A, threshold):.4f}, FN: {compute_FN(df_all_A, threshold):.4f}")
print(f"Dataset B - FP: {compute_FP(df_all_B, threshold):.4f}, FN: {compute_FN(df_all_B, threshold):.4f}")


Dataset A - FP: 0.0263, FN: 0.5196
Dataset B - FP: 0.0853, FN: 0.6538


In [6]:
thresholds_A = sorted(df_all_A["score"].unique().tolist() + [0, 1])
thresholds_B = sorted(df_all_B["score"].unique().tolist() + [0, 1])

# Dataset A
fps_A = {
    ths: compute_FP(df_all_A, ths)
    for ths in thresholds_A
}
fns_A = {
    ths: compute_FN(df_all_A, ths)
    for ths in thresholds_A
}

# Dataset B
fps_B = {
    ths: compute_FP(df_all_B, ths)
    for ths in thresholds_B
}
fns_B = {
    ths: compute_FN(df_all_B, ths)
    for ths in thresholds_B
}


In [7]:
import plotly.graph_objects as go

fig = go.Figure()

# Dataset A
fig.add_trace(go.Scatter(
    x=list(fps_A.values()),
    y=list(fns_A.values()),
    mode='lines+markers',
    name='Dataset A',
    marker=dict(
        size=2,
        color=thresholds_A,
        colorscale='Viridis',
        showscale=False,
    ),
    line=dict(color='royalblue', width=3),
    text=[f"Threshold: {ths:.4f}" for ths in thresholds_A],
    hovertemplate='<b>Dataset A</b><br><b>FPR:</b> %{x:.4f}<br><b>FNR:</b> %{y:.4f}<br>%{text}<extra></extra>'
))

# Dataset B
fig.add_trace(go.Scatter(
    x=list(fps_B.values()),
    y=list(fns_B.values()),
    mode='lines+markers',
    name='Dataset B',
    marker=dict(
        size=2,
        color=thresholds_B,
        colorscale='Plasma',
        showscale=False,
    ),
    line=dict(color='orangered', width=3),
    text=[f"Threshold: {ths:.4f}" for ths in thresholds_B],
    hovertemplate='<b>Dataset B</b><br><b>FPR:</b> %{x:.4f}<br><b>FNR:</b> %{y:.4f}<br>%{text}<extra></extra>'
))

fig.update_layout(
    title="ROC Curve - False Positive vs False Negative Rate (Datasets A & B)",
    xaxis_title="False Positive Rate",
    yaxis_title="False Negative Rate",
    xaxis=dict(range=[0, 1]),
    yaxis=dict(range=[0, 1]),
    hovermode='closest',
    template='plotly_white',
    width=800,
    height=700,
    font=dict(size=12),
    showlegend=True
)

fig.show()


### Area bajo la curva

In [8]:
def compute_auc(fps, fns):
    roc_points = sorted(zip(fps, fns), key=lambda p: p[0])
    fps_sorted, fns_sorted = zip(*roc_points)

    auc = 0
    for i in range(1, len(fps_sorted)):
        fp, fn = fps_sorted[i], fns_sorted[i]
        fp_, fn_ = fps_sorted[i-1], fns_sorted[i-1]
        
        auc += (fn + fn_) * 0.5 * (fp - fp_)

    return auc

auc_A = compute_auc(fps_A.values(), fns_A.values())
auc_B = compute_auc(fps_B.values(), fns_B.values())

print(f"AUC Dataset A: {auc_A:.4f}")
print(f"AUC Dataset B: {auc_B:.4f}")


AUC Dataset A: 0.8138
AUC Dataset B: 0.5726


### FP(FN=X) y umbral

In [9]:
def compute_fp_given_fn_x(fns, fps, target_fn):
    dif = [abs(fn - target_fn) for fn in fns.values()]
    closest_fn = sorted(zip(dif, fns.items()), key=lambda p: p[0])[0]
    thrs = closest_fn[1][0]

    return {
        "thrs": thrs,
        "fn": closest_fn[1][1],
        "fp": fps[thrs],
    }

x = 0.25
print(f"FP at FN={x} (Dataset A): {compute_fp_given_fn_x(fns_A, fps_A, x)}")
print(f"FP at FN={x} (Dataset B): {compute_fp_given_fn_x(fns_B, fps_B, x)}")


FP at FN=0.25 (Dataset A): {'thrs': 0.289883, 'fn': 0.2503496503496504, 'fp': 0.0}
FP at FN=0.25 (Dataset B): {'thrs': 0.532139, 'fn': 0.2503496503496504, 'fp': 0.0}


### FN(FP=X) y umbral

In [10]:
def compute_fn_given_fp_x(fns, fps, target_fp):
    dif = [abs(fp - target_fp) for fp in fps.values()]
    closest_fp = sorted(zip(dif, fps.items()), key=lambda p: p[0])[0]
    thrs = closest_fp[1][0]

    return {
        "thrs": thrs,
        "fn": fns[thrs],
        "fp": closest_fp[1][1],
    }

x = 0.25
print(f"FN at FP={x} (Dataset A): {compute_fn_given_fp_x(fns_A, fps_A, x)}")
print(f"FN at FP={x} (Dataset B): {compute_fn_given_fp_x(fns_B, fps_B, x)}")


FN at FP=0.25 (Dataset A): {'thrs': 0.044937, 'fn': 0.8363636363636364, 'fp': 0.25}
FN at FP=0.25 (Dataset B): {'thrs': 0.032287, 'fn': 0.8377622377622378, 'fp': 0.25}


### FN = FP y umbral


In [11]:
def compute_fn_eq_fp(fns, fps):
    dif = [abs(fp - fn) for fn, fp in zip(fns.values(), fps.values())]

    closest_fp = sorted(zip(dif, fps.items(), fns.items()), key=lambda p: p[0])[0]
    thrs = closest_fp[1][0]

    return {
        "thrs": thrs,
        "fn": closest_fp[2][1],
        "fp": closest_fp[1][1],
    }

print(f"FN = FP (Dataset A): {compute_fn_eq_fp(fns_A, fps_A)}")
print(f"FN = FP (Dataset B): {compute_fn_eq_fp(fns_B, fps_B)}")


FN = FP (Dataset A): {'thrs': 1.0, 'fn': 0.0, 'fp': 0.0}
FN = FP (Dataset B): {'thrs': 1.0, 'fn': 0.0, 'fp': 0.0}
